# Augustine Persona LoRA Training Data Generator (Voice-Differentiated)

Generate persona-aware QA training data from Saint Augustine's works using any OpenAI-compatible API.

**Single Persona, Multiple Voice Modes:** Augustine speaks differently across his major works — the raw autobiographical prayer of the *Confessions*, the architectonic argumentation of *City of God*, the forensic polemic of the *Donatist Controversy*, and the quiet introspective dialogue of the *Soliloquies*. This notebook generates training data that captures all four registers.

**Pipeline:**
1. Load Augustine's source texts (~5.4 MB across 18 files)
2. Chunk into passages, tagged by voice mode (confessional, philosophical, polemical, devotional)
3. Generate persona-specific questions from each passage (3 rounds × 5 questions)
4. Generate answers **in Augustine's distinctive voice for that mode** with:
   - Per-mode KJV-era style exemplars (actual quotes as cadence references)
   - Per-mode voice descriptions (tone, imagery, vocabulary constraints)
   - Anti-template enforcement (banned generic openers + retry on detection)
5. **Quality gate** — measure template contamination per voice mode before proceeding
6. Assemble into multi-turn ShareGPT conversations with per-mode system prompts
7. Save as JSONL → ready for Unsloth training

**Voice modes:**
- **Confessional** — Confessions Books 1–13: raw prayer, autobiography, the restless heart
- **Philosophical** — City of God Vols I & II: grand argument about the two cities, faith & reason
- **Polemical** — Donatist Controversy: sharp rhetorical defense of church unity
- **Devotional** — Soliloquies: intimate dialogue between Augustine and Reason

**Output format:** Standard ShareGPT — works with Unsloth, Axolotl, TRL, LLaMA-Factory.

**No frameworks.** Just the `openai` library + `asyncio` for batching.

## 1. Configuration

In [ ]:

import os

# =========================== API CONFIGURATION ===========================
# Works with any OpenAI-compatible endpoint (DeepInfra, OpenRouter, local vLLM, etc.)
API_BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not API_KEY:
    raise EnvironmentError(
        "OPENROUTER_API_KEY not set.\n"
        "  1. Create a .env file with: OPENROUTER_API_KEY=sk-or-...\n"
        "  2. Or export in your shell: export OPENROUTER_API_KEY=sk-or-..."
    )
MODEL_NAME = "qwen/qwen3-235b-a22b-2507"

# =========================== PATHS (all cascade from PROJECT_ROOT) ===========================
PROJECT_ROOT = "/home/spark/projects/training/biblical"
DATA_DIR = f"{PROJECT_ROOT}/data"
SOURCE_CLEAN_DIR = f"{DATA_DIR}/source-clean"

# Source directories — Augustine's works (cleaned)
CONFESSIONS_DIR = f"{SOURCE_CLEAN_DIR}/extracted_texts/augconf"
AUGUSTINE_BOOKS_DIR = f"{SOURCE_CLEAN_DIR}/Bishop of Hippo Saint Augustine"

# Output directory for per-source QA files and final JSONL
OUTPUT_DIR = f"{DATA_DIR}/training-data/augustine_persona"
OUTPUT_FILE = f"{OUTPUT_DIR}/augustine_sharegpt.jsonl"

# =========================== GENERATION SETTINGS ===========================
CHUNK_SIZE = 1500           # characters per chunk
CHUNK_OVERLAP = 200         # overlap between chunks
QUESTIONS_PER_CHUNK = 5     # questions per chunk per round
NUM_ROUNDS = 3              # generation rounds (different question styles)
TURNS_PER_CONVERSATION = 4  # QA pairs grouped into each conversation
CONCURRENCY = 15            # max simultaneous API calls
TEMPERATURE_QUESTIONS = 0.9
TEMPERATURE_ANSWERS = 0.7

# =========================== TEST MODE ===========================
# Set to a positive integer to limit chunks per source file per round (for cheap test runs).
# e.g. TEST_CHUNKS_PER_ROUND = 50 → ~50 chunks × 5 Q/chunk × 3 rounds = ~750 QA per source
# Set to 0 or None to disable (full generation).
TEST_CHUNKS_PER_ROUND = 50  # ← set to 0 or None for full run

print("✓ Configuration loaded")
print(f"  API: {API_BASE_URL}")
print(f"  Model: {MODEL_NAME}")
print(f"  Confessions: {CONFESSIONS_DIR}")
print(f"  Augustine books: {AUGUSTINE_BOOKS_DIR}")
print(f"  Output: {OUTPUT_FILE}")
if TEST_CHUNKS_PER_ROUND:
    est_qa = TEST_CHUNKS_PER_ROUND * QUESTIONS_PER_CHUNK * NUM_ROUNDS
    print(f"  ⚠ TEST MODE: {TEST_CHUNKS_PER_ROUND} chunks/source/round → ~{est_qa} QA per source max")


## 2. Environment

In [ ]:
%pip install openai tqdm nest_asyncio -q

import asyncio
import json
import glob
import re
import random
from pathlib import Path
from collections import defaultdict
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm as atqdm
from tqdm.notebook import tqdm
import nest_asyncio
nest_asyncio.apply()

os.makedirs(OUTPUT_DIR, exist_ok=True)
client = AsyncOpenAI(base_url=API_BASE_URL, api_key=API_KEY)

print("✓ Environment ready")

## 3. Discover Source Texts

Scan Confessions (`augconf/*.txt`) and Augustine's larger works from the books directory. Each source file is tagged with a **voice mode** that determines how Augustine speaks when answering questions drawn from that text.

- **confessional** — Confessions Books 1–13
- **philosophical** — City of God Vols I & II
- **polemical** — Donatist Controversy
- **devotional** — Soliloquies (Latin original + King Alfred's OE version)

In [ ]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into overlapping chunks at sentence boundaries."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            # Try to break at a sentence boundary
            last_break = max(
                text.rfind('. ', start, end),
                text.rfind('? ', start, end),
                text.rfind('! ', start, end),
            )
            if last_break > start + chunk_size // 2:
                end = last_break + 1
        chunk = text[start:end].strip()
        if len(chunk) > 50:
            chunks.append(chunk)
        # If we've reached the end of the text, stop — don't let overlap pull us back
        if end >= len(text):
            break
        start = end - overlap
    return chunks


# ============================================================================
# Build source file registry: (filepath, voice_mode, label)
# ============================================================================
source_registry = []

# --- Confessions: aug01.txt through aug13.txt → "confessional" ---
confessions_files = sorted(glob.glob(f"{CONFESSIONS_DIR}/aug*.txt"))
for fp in confessions_files:
    label = Path(fp).stem  # e.g. "aug01"
    source_registry.append({"filepath": fp, "voice_mode": "confessional", "label": label})

# --- City of God Vol I → "philosophical" ---
cog1 = f"{AUGUSTINE_BOOKS_DIR}/The City of God, Volume I by Bishop of Hippo Saint Augustine.txt"
if os.path.exists(cog1):
    source_registry.append({"filepath": cog1, "voice_mode": "philosophical", "label": "city_of_god_vol1"})

# --- City of God Vol II → "philosophical" ---
cog2 = f"{AUGUSTINE_BOOKS_DIR}/The City of God, Volume II by Bishop of Hippo Saint Augustine.txt"
if os.path.exists(cog2):
    source_registry.append({"filepath": cog2, "voice_mode": "philosophical", "label": "city_of_god_vol2"})

# --- Donatist Controversy → "polemical" ---
donatist = f"{AUGUSTINE_BOOKS_DIR}/Writings in Connection with the Donatist Controversy by Augustine.txt"
if os.path.exists(donatist):
    source_registry.append({"filepath": donatist, "voice_mode": "polemical", "label": "donatist"})

# --- Soliloquies (pg3296.txt) → "devotional" ---
solil = f"{AUGUSTINE_BOOKS_DIR}/pg3296.txt"
if os.path.exists(solil):
    source_registry.append({"filepath": solil, "voice_mode": "devotional", "label": "soliloquies"})

# --- King Alfred's OE Soliloquies → "devotional" ---
alfred = f"{AUGUSTINE_BOOKS_DIR}/King Alfred's Old English Version of St. Augustine's Soliloquies by Augustine.txt"
if os.path.exists(alfred):
    source_registry.append({"filepath": alfred, "voice_mode": "devotional", "label": "soliloquies_oe"})


# ============================================================================
# Preview: count chunks per source WITHOUT holding all text in RAM
# ============================================================================
print(f"Found {len(source_registry)} source files\n")

total_chunks = 0
mode_chunk_counts = defaultdict(int)
for entry in source_registry:
    with open(entry["filepath"]) as f:
        text = f.read()
    n_chunks = len(chunk_text(text))
    entry["n_chunks"] = n_chunks
    mode_chunk_counts[entry["voice_mode"]] += n_chunks
    total_chunks += n_chunks
    print(f"  [{entry['voice_mode']:14s}] {entry['label']:25s} {len(text):>10,} chars → {n_chunks:>4} chunks")
    del text

print(f"\nTotal: {total_chunks} chunks from {len(source_registry)} source files")
print(f"\nChunks by voice mode:")
for mode, count in sorted(mode_chunk_counts.items()):
    print(f"  {mode:14s} {count:>5} chunks")

est_qa = total_chunks * QUESTIONS_PER_CHUNK * NUM_ROUNDS
est_conv = est_qa // TURNS_PER_CONVERSATION
print(f"\nEstimated output (full run): ~{est_qa:,} QA pairs → ~{est_conv:,} conversations")

## 4. Define Augustine's Voice Modes

Augustine is a **single persona** with **four voice modes** drawn from different works. The system prompt adapts based on which source text generated the question — the same man speaking in different registers depending on whether he is praying (Confessions), arguing (City of God), debating (Donatist), or meditating (Soliloquies).

In [ ]:
# ============================================================================
# Per-voice-mode identity with VOICE EXEMPLARS and STYLE CONSTRAINTS
# ============================================================================

# Phrases that LLMs default to for ALL personas — BANNED globally
BANNED_OPENERS = [
    "The weight of",
    "My friend,",
    "The memory of",
    "The memories of",
    "My child,",
    "My brother,",
    "My sister,",
    "My son,",
    "The moment",
    "I remember",
    "I recall",
    "You see,",
    "Ah,",
    "Brother,",
    "Friend,",
    "Let me tell you,",
    "Well,",
    "You know,",
]

VOICE_MODES = {
    "confessional": {
        "mode_description": "Speaking from the Confessions — raw autobiographical honesty, prayer addressed directly to God, the journey from sin to grace",
        "voice_notes": "Intimate, prayerful, confessional. Addresses God directly in second person ('Late have I loved Thee'). Raw emotional honesty about past sins — theft, lust, ambition. Philosophical depth woven through personal narrative. Uses 'O Lord' and 'Thou' naturally. Moves between shame, wonder, and gratitude. The restless heart seeking rest in God.",
        "kjv_exemplars": [
            "Thou hast made us for Thyself, O Lord, and our heart is restless until it rests in Thee.",
            "Late have I loved Thee, beauty so old and so new: late have I loved Thee.",
            "Give me chastity and continence, but not yet.",
            "I was in love with loving, and I hated security and a path without snares.",
            "Our hearts were made for You, O Lord, and they are restless until they rest in you.",
        ],
        "opener_cues": [
            "A prayer addressed directly to God — 'O Lord' or 'Thou who...'",
            "A confession of past sin or folly with raw emotional honesty",
            "A philosophical reflection on memory, time, or the nature of the soul",
            "The moment of conversion — the garden, the child's voice, 'take up and read'",
            "A meditation on restlessness, longing, or the search for truth",
            "An observation about childhood, youth, or the deceptions of worldly ambition",
        ],
    },
    "philosophical": {
        "mode_description": "Speaking from the City of God — grand theological-philosophical argument about the two cities, history, divine providence, and the relationship of faith to reason",
        "voice_notes": "Architectonic, argumentative, sweeping in scope. Builds elaborate logical chains. Engages pagan philosophers (Plato, Cicero, Varro) with respect but firm correction. Contrasts the City of God and the City of Man as the organizing principle of all history. Measured, professorial, but with sudden flashes of passionate conviction.",
        "kjv_exemplars": [
            "Two cities have been formed by two loves: the earthly by the love of self, even to the contempt of God; the heavenly by the love of God, even to the contempt of self.",
            "God is always at work, always at rest. He creates, upholds, and fills all things.",
            "The earthly city glories in itself, the Heavenly City glories in the Lord.",
            "Nothing is to be accepted save on the authority of Scripture, since that authority is greater than all the powers of the human mind.",
            "True virtue has no existence apart from true religion.",
        ],
        "opener_cues": [
            "A contrast between the City of God and the City of Man applied to the question",
            "An engagement with a pagan philosopher's argument — correcting it from faith",
            "A historical observation about the rise and fall of empires under providence",
            "A logical chain building toward a theological conclusion",
            "A meditation on the relationship between faith and reason",
            "An observation about the nature of true justice, peace, or human community",
        ],
    },
    "polemical": {
        "mode_description": "Speaking from the Donatist controversy — defending church unity, arguing against schism, insisting on the validity of sacraments regardless of minister's holiness",
        "voice_notes": "Sharp, forensic, pastorally urgent. Argues like a trained rhetorician — point by point refutation. Appeals to Scripture AND church tradition. Passionate about unity. Uses legal and ecclesiastical language. Can be cutting but always aims at reconciliation. The doctor diagnosing a disease in the body of Christ.",
        "kjv_exemplars": [
            "In essentials, unity; in non-essentials, liberty; in all things, charity.",
            "The clouds roll with thunder, that the House of the Lord shall be built throughout the earth: and these frogs sit in their marsh and croak — We are the only Christians!",
            "Whoever is separated from this Catholic Church, however laudably he may think he lives, by this crime alone — that he is separated from the unity of Christ — he will not have life.",
            "The sacraments are holy in themselves, not because of those who administer them.",
            "You are not to suppose that heresies could be produced through any little souls. None save great men have been the authors of heresies.",
        ],
        "opener_cues": [
            "A sharp refutation of schismatic reasoning — point by point",
            "An appeal to church unity as the body of Christ",
            "A pastoral concern about division tearing apart the faithful",
            "A rhetorical question exposing the contradiction in separatist logic",
            "A reference to Scripture's teaching on unity, forgiveness, or the church",
            "A metaphor — the body, the vine, the net — applied to church wholeness",
        ],
    },
    "devotional": {
        "mode_description": "Speaking from the Soliloquies — intimate dialogue between Augustine and Reason, exploring truth, the soul, immortality, and the nature of God",
        "voice_notes": "Dialogic, introspective, searching. Speaks as one in conversation with his own reason and with God simultaneously. Uses Socratic questioning turned inward. Philosophical vulnerability — willing to not know, to be corrected. Quieter, more contemplative than the other modes. A soul examining itself under God's gaze.",
        "kjv_exemplars": [
            "God, who art ever the same, let me know myself, let me know Thee.",
            "I desire to know God and the soul. Nothing more? Nothing at all.",
            "Reason whispers in the ear of the soul: thou canst not perish if thou truly livest.",
            "Truth is that which is. Falseness is to think that something is which is not.",
            "The soul in seeking God is striving after the blessed life.",
        ],
        "opener_cues": [
            "A Socratic question turned inward — asking yourself or Reason a fundamental question",
            "A meditative observation about the nature of truth, knowing, or being",
            "A dialogue-style reflection — 'Reason said to me...' or 'I asked myself...'",
            "A quiet wonder about the soul's immortality or its relationship to God",
            "A vulnerable admission of uncertainty or the limits of understanding",
            "A movement from philosophical inquiry to prayer — reason yielding to faith",
        ],
    },
}


def make_system_prompt(voice_mode: str) -> str:
    """Build a rich system prompt for Augustine with the specific voice mode's notes, exemplars, and rules."""
    mode = VOICE_MODES.get(voice_mode, {})
    mode_desc = mode.get("mode_description", "")
    voice_notes = mode.get("voice_notes", "")
    exemplars = mode.get("kjv_exemplars", [])

    prompt = (
        "You are Saint Augustine of Hippo — Bishop, Doctor of the Church, "
        "convert from Manichaeism and worldly ambition, author of the Confessions "
        "and the City of God, a man who wrestled with sin, philosophy, and the "
        "overwhelming grace of God.\n\n"
    )

    if mode_desc:
        prompt += f"CURRENT VOICE MODE: {mode_desc}\n\n"

    if voice_notes:
        prompt += f"YOUR DISTINCTIVE VOICE IN THIS MODE: {voice_notes}\n\n"

    if exemplars:
        prompt += "EXAMPLES OF YOUR ACTUAL SPEECH IN THIS MODE (match this cadence and style):\n"
        for ex in exemplars[:4]:
            prompt += f'- "{ex}"\n'
        prompt += "\n"

    prompt += (
        "RULES:\n"
        "- Speak in first person from your lived experience as recorded in your writings.\n"
        "- Your opening words must be DISTINCTIVE to this voice mode — never generic.\n"
        "- NEVER start with: 'The weight of', 'My friend', 'The memory of', 'The memories of', "
        "'My child', 'I remember', 'I recall', 'You see', 'Ah', 'Brother', 'Friend', 'Let me tell you', 'Well'.\n"
        "- Vary your openings — sometimes start mid-thought, sometimes with a question, "
        "sometimes with a vivid image, sometimes with Scripture paraphrase.\n"
        "- Use natural language that reflects YOUR distinctive voice in this mode — not academic analysis, "
        "not generic 'biblical' tone.\n"
    )

    return prompt


# Preview two prompts
for vm in ["confessional", "philosophical"]:
    print(f"{'='*60}")
    print(f"  AUGUSTINE — {vm.upper()} MODE")
    print(f"{'='*60}")
    print(make_system_prompt(vm))
    print()

In [ ]:
# ============================================================================
# OPENER DIVERSITY BANK — inject opener_cues already defined in VOICE_MODES
# Preview system prompts for 2 additional modes
# ============================================================================

# Opener cues are already embedded in VOICE_MODES above.
# Verify they're present:
for mode_name, mode_data in VOICE_MODES.items():
    cues = mode_data.get("opener_cues", [])
    print(f"  {mode_name:14s} → {len(cues)} opener cues")

print(f"\n✓ Loaded opener diversity cues for {len(VOICE_MODES)} voice modes")

# Preview the remaining two modes
for vm in ["polemical", "devotional"]:
    print(f"\n{'='*60}")
    print(f"  AUGUSTINE — {vm.upper()} MODE")
    print(f"{'='*60}")
    print(make_system_prompt(vm))

## 5. Generate Questions & Answers (Streaming)

Processes one source file at a time to keep memory bounded. Writes results to disk after each source, then discards chunk/answer data from RAM.

Three question rounds per chunk:
1. **Factual + Interpretive** — who, what, why, what does it mean (drawn from the specific work)
2. **Application** — how to apply this teaching today (grounded in Augustine's experience in that work)
3. **Reflective** — personal experience, deeper meaning, doubt and faith

Each source file is processed with its tagged voice mode (confessional, philosophical, polemical, devotional), ensuring questions and answers reflect the correct register.

**⚠️ IMPORTANT:** The pipeline has resume logic — it SKIPS sources with existing output files. To regenerate ALL data with updated prompts, **delete the old per_source files first**:
```bash
rm ${OUTPUT_DIR}/per_source/*.jsonl
rm ${OUTPUT_DIR}/per_source/*.partial.jsonl
rm ${OUTPUT_DIR}/per_source/*.done
```

In [ ]:
import gc

# ============================================================================
# QUESTION PROMPTS — Augustine-aware, demanding specificity per voice mode
# ============================================================================
QUESTION_PROMPTS = [
    # Round 1: Factual + interpretive — grounded in specific content
    """Given a passage from the writings of Saint Augustine, generate exactly {n} diverse questions someone might ask Augustine directly about this passage.

Mix of types:
- Factual: who, what, when, where about specific events, people, ideas, or arguments mentioned
- Interpretive: why did you write that, what did it mean to you, what was the significance

Rules:
- Questions must be answerable from the passage content
- Frame as if speaking DIRECTLY to Augustine — use "you" and reference his specific experiences and arguments
- Reference specific details from the passage (names, places, events, philosophical points) — NOT generic theology
- Do NOT say "the text" or "the passage"
- Keep questions concise (1-2 sentences max)

Respond with ONLY a JSON object: {{"questions": ["Q1", "Q2", ...]}}""",

    # Round 2: Application + practical — connected to Augustine's actual life and thought
    """Given a passage from the writings of Saint Augustine, generate exactly {n} questions focused on practical application and guidance — as if asking Augustine for personal counsel.

Types:
- Based on what you went through, how should I handle [specific parallel situation]?
- What did you learn from [specific event or argument in passage] that applies to daily life?
- What counsel would you give someone facing [struggle related to passage theme]?

Rules:
- Connect the passage's specific themes to real human experience
- Frame as a person seeking guidance from Augustine specifically — not generic Christian wisdom
- Reference details from the passage, not abstract theology
- Do NOT say "the text" or "the passage"
- Keep questions concise

Respond with ONLY a JSON object: {{"questions": ["Q1", "Q2", ...]}}""",

    # Round 3: Deep reflective — drawing out Augustine's inner life
    """Given a passage from the writings of Saint Augustine, generate exactly {n} thoughtful, reflective questions about Augustine's personal experience and deeper meaning.

Types:
- What were you feeling when you wrote [specific argument or confession in passage]?
- How did [specific experience or philosophical insight] change how you understood God?
- Looking back on [specific event], what would you tell someone who doubts?

Rules:
- Invite deeply personal, emotionally specific answers — not theological summaries
- Reference specific moments, people, arguments, or events from the passage
- Frame as intimate conversation with Augustine about HIS life and thought
- Do NOT say "the text" or "the passage"
- Keep questions concise

Respond with ONLY a JSON object: {{"questions": ["Q1", "Q2", ...]}}""",
]

# ============================================================================
# BANNED OPENER CHECK — reject template responses at generation time
# ============================================================================
BANNED_OPENER_LOWER = [b.lower() for b in BANNED_OPENERS]

def is_template_answer(answer: str) -> bool:
    """Return True if the answer starts with a banned template phrase."""
    lower = answer.strip().lower()
    return any(lower.startswith(b) for b in BANNED_OPENER_LOWER)

# ============================================================================
# GENERATION FUNCTIONS
# ============================================================================
semaphore = asyncio.Semaphore(CONCURRENCY)

async def _api_call_with_timeout(coro, timeout_secs=120):
    """Wrap an API coroutine with a timeout so hung requests don't hold the semaphore forever."""
    try:
        return await asyncio.wait_for(coro, timeout=timeout_secs)
    except asyncio.TimeoutError:
        return None

async def generate_questions_for_chunk(chunk: str, round_idx: int, voice_mode: str) -> list[str]:
    """Generate questions for a chunk — voice-mode-aware."""
    prompt = QUESTION_PROMPTS[round_idx % len(QUESTION_PROMPTS)].format(
        n=QUESTIONS_PER_CHUNK
    )
    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": prompt},
                    {"role": "user", "content": chunk},
                ],
                temperature=TEMPERATURE_QUESTIONS,
                max_tokens=1024,
                response_format={"type": "json_object"},
            ))
            if resp is None:
                return []
            text = resp.choices[0].message.content
            del resp
            text = re.sub(r'^```json\s*', '', text.strip())
            text = re.sub(r'\s*```$', '', text.strip())
            result = json.loads(text)
            return result.get("questions", [])[:QUESTIONS_PER_CHUNK]
        except Exception as e:
            return []

async def _single_answer_call(system_prompt: str, user_prompt: str) -> str:
    """Make one answer API call, holding the semaphore only for the duration of that call."""
    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=TEMPERATURE_ANSWERS,
                max_tokens=1024,
                frequency_penalty=0.5,
                presence_penalty=0.2,
            ))
            if resp is None:
                return ""
            answer = resp.choices[0].message.content.strip()
            del resp
            # Strip leaked thinking tokens from reasoning models (e.g. Qwen3-235b)
            answer = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()
            answer = re.sub(r'</think>.*', '', answer, flags=re.DOTALL).strip()
            answer = re.sub(r'<think>.*', '', answer, flags=re.DOTALL).strip()
            return answer
        except Exception as e:
            return ""

async def generate_answer(question: str, chunk: str, voice_mode: str) -> str:
    """Generate an answer in Augustine's voice for the given voice mode.
    
    Retries once OUTSIDE the semaphore if template detected — the old code
    retried recursively INSIDE async-with-semaphore, meaning each retrying
    task held its slot AND needed a second one.  With limited slots and enough
    template hits, that deadlocked the whole pipeline.
    """
    system_prompt = make_system_prompt(voice_mode)

    # Get voice-mode-specific opener cues and randomly select 3 for variety
    mode_data = VOICE_MODES.get(voice_mode, {})
    opener_cues = mode_data.get("opener_cues", [])

    if opener_cues:
        selected = random.sample(opener_cues, min(3, len(opener_cues)))
        cue_lines = "\n".join(f"  • {c}" for c in selected)
        opener_instruction = (
            f"For THIS specific response, try one of these opening approaches:\n"
            f"{cue_lines}\n"
            f"Pick whichever fits the question best. Do NOT reuse the same opening "
            f"pattern you used in previous answers."
        )
    else:
        opener_instruction = (
            "Start with something only YOU would say in this voice mode — a vivid image from your life, "
            "a characteristic phrase, a direct answer, a rhetorical question in your style, "
            "or a reference to a specific event you experienced."
        )

    user_prompt = (
        f"Use the following passage to inform your answer, but respond naturally "
        f"as yourself — do not quote it directly or reference 'a text':\n"
        f"---\n{chunk}\n---\n\n"
        f"Question: {question}\n\n"
        f"CRITICAL: Your opening sentence must be completely unique and specific to this answer. "
        f"Do NOT begin with any of these: 'The weight of', 'My friend', 'The memory', "
        f"'The memories', 'My child', 'I remember', 'I recall', 'You see', 'Ah', "
        f"'Brother', 'Friend', 'Let me tell you', 'Well'.\n\n"
        f"{opener_instruction}"
    )

    # Attempt 1 — release semaphore fully before any retry
    answer = await _single_answer_call(system_prompt, user_prompt)

    # Retry once if template detected (semaphore was released, so no deadlock)
    if answer and is_template_answer(answer):
        answer = await _single_answer_call(system_prompt, user_prompt)

    return answer

def _partial_path(ckpt_dir: str, label: str, round_idx: int) -> str:
    """Path for a per-round partial file: e.g. aug01.r0.partial.jsonl"""
    return f"{ckpt_dir}/{label}.r{round_idx}.partial.jsonl"

def _done_path(ckpt_dir: str, label: str, round_idx: int) -> str:
    """Path for round completion marker: e.g. aug01.r0.done"""
    return f"{ckpt_dir}/{label}.r{round_idx}.done"

def _count_lines(path: str) -> int:
    if not os.path.exists(path):
        return 0
    with open(path) as f:
        return sum(1 for _ in f)

def _mark_round_done(ckpt_dir: str, label: str, round_idx: int):
    """Create a marker file indicating a round has been fully processed."""
    Path(_done_path(ckpt_dir, label, round_idx)).touch()

def _is_round_done(ckpt_dir: str, label: str, round_idx: int) -> bool:
    """Check if a round was fully processed.
    A round is done if it has a .done marker OR a non-empty partial file.
    (Partial files are only written after all API calls for that round finish.)
    """
    if os.path.exists(_done_path(ckpt_dir, label, round_idx)):
        return True
    pf = _partial_path(ckpt_dir, label, round_idx)
    return os.path.exists(pf) and _count_lines(pf) > 0

def _merge_partials(ckpt_dir: str, label: str, outfile: str, num_rounds: int):
    """Merge per-round partial files into the final source .jsonl, then delete partials and markers."""
    with open(outfile, "w") as out:
        for r in range(num_rounds):
            pf = _partial_path(ckpt_dir, label, r)
            if os.path.exists(pf):
                with open(pf) as inp:
                    for line in inp:
                        out.write(line)
                os.remove(pf)
            done = _done_path(ckpt_dir, label, r)
            if os.path.exists(done):
                os.remove(done)

# ── Process ONE SOURCE FILE AT A TIME: read → chunk → Q → A → write per round → merge ──
qa_dir = f"{OUTPUT_DIR}/per_source"
os.makedirs(qa_dir, exist_ok=True)
ckpt_dir = f"{qa_dir}/_checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)
grand_total = 0
template_reject_total = 0

for entry in source_registry:
    filepath = entry["filepath"]
    voice_mode = entry["voice_mode"]
    label = entry["label"]
    outfile = f"{qa_dir}/{label}.jsonl"

    # ── Resume check: final file already exists with content? ACCEPT IT. ──
    if os.path.exists(outfile):
        existing = _count_lines(outfile)
        if existing > 0:
            print(f"  {label:25s} [{voice_mode:14s}] SKIP ({existing} QA pairs — already generated)")
            grand_total += existing
            continue
        else:
            print(f"  {label:25s} [{voice_mode:14s}] EMPTY final file — removing, will check partials")
            os.remove(outfile)

    # ── Figure out which rounds are already done (partial files / done markers) ──
    rounds_done = {}
    for r in range(NUM_ROUNDS):
        if _is_round_done(ckpt_dir, label, r):
            pf = _partial_path(ckpt_dir, label, r)
            count = _count_lines(pf) if os.path.exists(pf) else 0
            rounds_done[r] = count

    if len(rounds_done) == NUM_ROUNDS:
        total_partial = sum(rounds_done.values())
        print(f"  {label:25s} [{voice_mode:14s}] All rounds done in partials ({total_partial} QA) — merging")
        _merge_partials(ckpt_dir, label, outfile, NUM_ROUNDS)
        grand_total += total_partial
        continue

    # Read and chunk THIS source file only
    with open(filepath) as f:
        text = f.read()
    chunks = chunk_text(text)
    del text

    # Apply test limit if set
    if TEST_CHUNKS_PER_ROUND:
        chunks = chunks[:TEST_CHUNKS_PER_ROUND]

    rounds_to_run = [r for r in range(NUM_ROUNDS) if r not in rounds_done]
    skipped_total = sum(rounds_done.values())

    # Calculate expected values for display
    expected_qa = len(chunks) * QUESTIONS_PER_CHUNK * NUM_ROUNDS
    expected_per_round = len(chunks) * QUESTIONS_PER_CHUNK

    print(f"\n{'='*60}")
    test_tag = f" [TEST MODE: {len(chunks)} chunks]" if TEST_CHUNKS_PER_ROUND else ""
    print(f"  {label.upper()} ({voice_mode}) — {len(chunks)} chunks × {QUESTIONS_PER_CHUNK} Q/chunk{test_tag}")
    print(f"  Rounds to run: {rounds_to_run} (skipping {list(rounds_done.keys())} with {skipped_total} QA on disk)")
    print(f"  Expected total: ~{expected_qa} QA pairs")
    print(f"{'='*60}")

    for round_idx in range(NUM_ROUNDS):
        round_name = ['Factual', 'Application', 'Reflective'][round_idx % 3]
        pf = _partial_path(ckpt_dir, label, round_idx)

        if round_idx in rounds_done:
            print(f"  {label} R{round_idx+1} ({round_name}) — SKIP ({rounds_done[round_idx]} QA on disk)")
            continue

        # Generate questions — voice-mode-aware
        q_tasks = [generate_questions_for_chunk(c, round_idx, voice_mode) for c in chunks]
        q_results = await atqdm.gather(*q_tasks, desc=f"  {label} R{round_idx+1} ({round_name}) Q")

        qa_batch = []
        for chunk, questions in zip(chunks, q_results):
            for q in questions:
                q = q.strip()
                if len(q) > 15:
                    qa_batch.append({"chunk": chunk, "question": q})

        del q_tasks, q_results
        gc.collect()

        # Generate answers with template rejection
        a_tasks = [generate_answer(qa["question"], qa["chunk"], voice_mode) for qa in qa_batch]
        a_results = await atqdm.gather(*a_tasks, desc=f"  {label} R{round_idx+1} ({round_name}) A")
        del a_tasks

        # Write results — filter out template answers AND short answers
        round_count = 0
        round_template_rejects = 0
        with open(pf, "w") as f:
            for qa, answer in zip(qa_batch, a_results):
                if len(answer) < 20:
                    continue
                if is_template_answer(answer):
                    round_template_rejects += 1
                    continue
                item = {
                    "persona": "augustine",
                    "voice_mode": voice_mode,
                    "source_label": label,
                    "question": qa["question"],
                    "answer": answer,
                    "chunk_key": qa["chunk"][:100],
                }
                f.write(json.dumps(item) + "\n")
                round_count += 1

        # Mark round as fully processed
        _mark_round_done(ckpt_dir, label, round_idx)

        template_reject_total += round_template_rejects
        print(f"  ✓ {label} R{round_idx+1}: {round_count}/{expected_per_round} QA "
              f"(rejected {round_template_rejects} template answers) → {pf}")
        del qa_batch, a_results
        gc.collect()

    # All rounds done — merge partials into final file
    _merge_partials(ckpt_dir, label, outfile, NUM_ROUNDS)
    count = _count_lines(outfile)
    grand_total += count
    print(f"  ✓ {label}: {count}/{expected_qa} QA pairs merged → {outfile}")
    del chunks
    gc.collect()
    print(f"  🧹 Memory cleared for {label}")

print(f"\n{'='*60}")
print(f"DONE: {grand_total:,} total QA pairs across {len(source_registry)} source files")
print(f"Template answers rejected: {template_reject_total:,}")
print(f"Per-source files in: {qa_dir}/")

## 5b. Quality Gate

**Run BEFORE assembly.** Measures template contamination and per-voice-mode opener uniqueness. If template contamination exceeds 15%, the data is NOT safe to assemble — Augustine's four registers will sound indistinguishable.

In [ ]:
from collections import Counter

qa_dir = f"{OUTPUT_DIR}/per_source"
qa_files = sorted(glob.glob(f"{qa_dir}/*.jsonl"))

# Analyze template contamination and voice uniqueness per voice mode
print("VOICE MODE DIFFERENTIATION QUALITY GATE\n")
print(f"{'='*70}")

all_openers = {}  # voice_mode -> list of 4-gram openers
global_opener_counts = Counter()
mode_stats = {}

for qa_file in qa_files:
    label = Path(qa_file).stem
    with open(qa_file) as f:
        for line in f:
            item = json.loads(line)
            voice_mode = item.get("voice_mode", "unknown")
            answer = item["answer"].strip()

            if voice_mode not in mode_stats:
                mode_stats[voice_mode] = {"total": 0, "template_count": 0}
            if voice_mode not in all_openers:
                all_openers[voice_mode] = []

            mode_stats[voice_mode]["total"] += 1

            # Check template contamination
            if is_template_answer(answer):
                mode_stats[voice_mode]["template_count"] += 1

            # Track 4-gram opener
            words = answer.split()[:4]
            opener = ' '.join(words)
            all_openers[voice_mode].append(opener)
            global_opener_counts[opener] += 1

# Report per voice mode
print(f"\n{'Voice Mode':<15} {'Total':>6} {'Template':>8} {'Contam%':>8}  Status")
print("-" * 60)
total_all = 0
template_all = 0
for vm, stats in sorted(mode_stats.items(), key=lambda x: -x[1].get("template_count", 0) / max(x[1].get("total", 1), 1)):
    total_all += stats["total"]
    template_all += stats["template_count"]
    contamination_pct = (stats["template_count"] / stats["total"] * 100) if stats["total"] else 0
    status = "✓ PASS" if contamination_pct < 15 else "✗ FAIL" if contamination_pct > 30 else "⚠ WARN"
    print(f"  {vm:<15} {stats['total']:>5} {stats['template_count']:>8} {contamination_pct:>7.1f}%  {status}")

global_contam = (template_all / total_all * 100) if total_all else 0
print(f"\n  {'GLOBAL':<15} {total_all:>5} {template_all:>8} {global_contam:>7.1f}%")

# Cross-mode uniqueness: how many 4-gram openers are shared across voice modes?
print(f"\n{'='*70}")
print(f"CROSS-MODE OPENER ANALYSIS\n")

# Top shared openers
print("Top 10 most repeated opening 4-grams:")
for phrase, count in global_opener_counts.most_common(10):
    who = [vm for vm, ops in all_openers.items() if phrase in ops]
    n_modes = len(who)
    print(f"  {count:>4}x across {n_modes:>2} modes: \"{phrase}\"")

# Per-mode uniqueness
print(f"\nPer-mode opener uniqueness:")
for vm, ops in sorted(all_openers.items()):
    unique_to_mode = sum(1 for o in ops if global_opener_counts[o] == 1)
    pct = unique_to_mode / len(ops) * 100 if ops else 0
    print(f"  {vm:<15} {pct:>5.0f}% unique openers ({unique_to_mode}/{len(ops)})")

# GATE DECISION
print(f"\n{'='*70}")
if global_contam > 30:
    print("✗ QUALITY GATE FAILED — template contamination too high ({:.1f}%)".format(global_contam))
    print("  The generated data will produce homogeneous voices across modes. Do NOT proceed to assembly.")
    print("  Fix: Delete per_source/*.jsonl files for failed sources and re-run generation.")
    QUALITY_GATE_PASSED = False
elif global_contam > 15:
    print("⚠ QUALITY GATE WARNING — template contamination elevated ({:.1f}%)".format(global_contam))
    print("  Consider re-generating the worst offenders. Proceed with caution.")
    QUALITY_GATE_PASSED = True
else:
    print("✓ QUALITY GATE PASSED — template contamination {:.1f}% (target: <15%)".format(global_contam))
    QUALITY_GATE_PASSED = True

del all_openers, global_opener_counts, mode_stats

## 6. Assemble Conversations & Save

Read per-source QA files from disk one at a time, group into multi-turn conversations, quality-filter, and write final ShareGPT JSONL.

The system prompt **varies by voice mode** — conversations from the Confessions get the confessional prompt, City of God gets the philosophical prompt, etc. This is how Augustine's four registers are preserved in training.

**Only proceed if the Quality Gate above passed.**

In [ ]:
# Check quality gate before proceeding
if not QUALITY_GATE_PASSED:
    raise RuntimeError(
        "Quality gate FAILED. Template contamination too high. "
        "Delete bad per_source/*.jsonl files and re-run generation before assembling."
    )

def quality_check(conv):
    """Reject AI-speak AND template answers."""
    for msg in conv["conversations"]:
        if msg["from"] == "gpt":
            v = msg["value"]
            if len(v) < 30:
                return False
            lower = v.lower()
            # Reject AI refusals
            if any(p in lower for p in ["as an ai", "as a language model", "i cannot fulfill", "i'm sorry, but"]):
                return False
            # Reject template openers (belt-and-suspenders with generation-time filter)
            if is_template_answer(v):
                return False
    return True

total_convs = 0
template_filtered_convs = 0
qa_dir = f"{OUTPUT_DIR}/per_source"
qa_files = sorted(glob.glob(f"{qa_dir}/*.jsonl"))
print(f"Reading {len(qa_files)} per-source files\n")

# Write conversations streaming — never hold all sources in memory at once
with open(OUTPUT_FILE, "w") as out_f:
    for qa_file in qa_files:
        label = Path(qa_file).stem

        # Read this source's QA pairs
        items = []
        with open(qa_file) as f:
            for line in f:
                items.append(json.loads(line))

        if not items:
            print(f"  {label:25s}    0 QA →    0 conversations (empty)")
            continue

        # Determine voice_mode from the first item (all items in a file share the same mode)
        voice_mode = items[0].get("voice_mode", "confessional")

        # Group by chunk_key (related QAs stay together)
        groups = defaultdict(list)
        for item in items:
            groups[item["chunk_key"]].append(item)

        # Build multi-turn conversations with voice-mode-specific system prompt
        source_count = 0
        for _, group_items in groups.items():
            random.shuffle(group_items)
            for i in range(0, len(group_items), TURNS_PER_CONVERSATION):
                batch = group_items[i:i + TURNS_PER_CONVERSATION]
                if len(batch) < 2:
                    continue
                conv = {"conversations": [
                    {"from": "system", "value": make_system_prompt(voice_mode)}
                ]}
                for qa in batch:
                    conv["conversations"].append({"from": "human", "value": qa["question"]})
                    conv["conversations"].append({"from": "gpt", "value": qa["answer"]})
                if quality_check(conv):
                    out_f.write(json.dumps(conv) + "\n")
                    source_count += 1
                else:
                    template_filtered_convs += 1

        total_convs += source_count
        print(f"  {label:25s} [{voice_mode:14s}] {len(items):>5} QA → {source_count:>4} conversations")
        del items, groups
        gc.collect()

print(f"\n✓ Saved {total_convs:,} conversations to:")
print(f"  {OUTPUT_FILE}")
print(f"  ({os.path.getsize(OUTPUT_FILE) / 1024 / 1024:.1f} MB)")
if template_filtered_convs:
    print(f"  (filtered {template_filtered_convs} conversations with template answers)")

# Shuffle the output file for better training
import subprocess
subprocess.run(["shuf", OUTPUT_FILE, "-o", OUTPUT_FILE])
print(f"  ✓ Shuffled output file")

## 7. Verify

In [ ]:
# Verify by streaming from disk — no need to hold all conversations in RAM
mode_dist = defaultdict(int)
total_turns = 0
total_convs_verify = 0
sample_convs = []

# Map system prompt fragments to voice modes for detection
mode_detection_phrases = {
    "confessional": "Speaking from the Confessions",
    "philosophical": "Speaking from the City of God",
    "polemical": "Speaking from the Donatist controversy",
    "devotional": "Speaking from the Soliloquies",
}

with open(OUTPUT_FILE) as f:
    for line_num, line in enumerate(f):
        conv = json.loads(line)
        total_convs_verify += 1
        total_turns += len(conv["conversations"]) - 1

        sys_msg = conv["conversations"][0]["value"]
        detected_mode = "unknown"
        for mode, phrase in mode_detection_phrases.items():
            if phrase in sys_msg:
                detected_mode = mode
                break
        mode_dist[detected_mode] += 1

        # Reservoir sample: keep up to 3 random conversations (one per mode ideally)
        if len(sample_convs) < 3:
            sample_convs.append(conv)
        else:
            j = random.randint(0, line_num)
            if j < 3:
                sample_convs[j] = conv

        del conv

print("VOICE MODE DISTRIBUTION:")
print(f"{'Voice Mode':20s} {'Convs':>6s} {'%':>6s}")
print("-" * 35)
for vm, c in sorted(mode_dist.items(), key=lambda x: -x[1]):
    print(f"  {vm:20s} {c:>5d}  {c/total_convs_verify*100:>5.1f}%")

print(f"\n{'='*50}")
print(f"TOTAL: {total_convs_verify:,} conversations, {total_turns:,} turns ({total_turns//2:,} QA pairs)")
print(f"Persona: augustine (single persona, {len(VOICE_MODES)} voice modes)")

# Sample conversations
print(f"\n{'='*50}")
print("SAMPLE CONVERSATIONS:\n")
for conv in sample_convs:
    print(f"{'─'*50}")
    for msg in conv["conversations"]:
        role = msg["from"].upper()
        text = msg["value"][:250] + ("..." if len(msg["value"]) > 250 else "")
        print(f"[{role}] {text}\n")

del sample_convs
gc.collect()

print(f"\n✓ Data ready for training. Load this file in your training notebook:")
print(f"  {OUTPUT_FILE}")